# 04 A549 FigS6H Threshold Exploration

This notebook explores **Figure S6H metastatic phenotype cutoffs** while keeping the canonical clone enrichment assignments fixed from `03_A549_FigS6_Supplementary.ipynb`.

- It does **not** overwrite the production Figure S6 exports.
- It reads the canonical clone-level export from notebook 03.
- It writes sweep tables and candidate preview figures under `04_FigS6H_Threshold_Exploration`.
- Historical lineage-level thresholds from the older `Replot_TreeMet_LG.ipynb` (`weak=0.02`, `high=0.1`) are recorded as reference only; they were originally chosen manually after visually inspecting the old `combined_met_rates.pdf` lineage-level plot, not by formal S6H optimization.
- A collaborator-guided expectation that the overall `Highly metastatic` fraction may be around 20-25% is tracked here as a soft reference, not a hard theoretical constraint.


In [ ]:
# Environment + determinism
import os

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, fisher_exact

sns.set_theme(style='whitegrid', context='paper')

In [ ]:
# Resolve repo-local paths
CWD = Path.cwd().resolve()
REPO_ROOT = None
for p in [CWD, *CWD.parents]:
    if (p / 'code' / 'figure6' / 'notebooks').exists() and (p / 'data' / 'figure6').exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root for dkfz_skeleton figure6 workflow.')

SCRIPTS_DIR = REPO_ROOT / 'code' / 'figure6' / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

from project_paths import get_notebook_paths

PATHS = get_notebook_paths('04_FigS6H_Threshold_Exploration', REPO_ROOT)
FIG3_PLOT_DIR = PATHS.common.data_dir / 'derived' / '03_FigS6_Supplementary' / 'plot_data'
DIR_PLOT_DATA = PATHS.save_dir / 'plot_data'
DIR_FIG_SAVE = PATHS.figure_dir
DIR_PLOT_DATA.mkdir(parents=True, exist_ok=True)
DIR_FIG_SAVE.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('INPUT_PLOT_DIR:', FIG3_PLOT_DIR)
print('SAVE_DIR:', PATHS.save_dir)
print('FIG_DIR:', DIR_FIG_SAVE)

In [ ]:
def my_savefig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    print('[Saved]', path)
    if path.suffix.lower() == '.pdf':
        png_path = path.with_suffix('.png')
        plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
        print('[Saved]', png_path)

In [ ]:
# Inputs exported by 03_A549_FigS6_Supplementary.ipynb
IN_CLONE = FIG3_PLOT_DIR / 'FigS6H_clone_enrichment_summary.csv'
IN_SITE = FIG3_PLOT_DIR / 'FigS6H_site_counts_eligible.csv'
IN_METADATA = FIG3_PLOT_DIR / 'FigS6_metadata.json'

if not IN_CLONE.exists():
    raise FileNotFoundError(f'Missing {IN_CLONE}. Run notebook 03 first.')

clone = pd.read_csv(IN_CLONE)
clone = clone[clone['enriched'].isin(['LOY', 'ROY'])].copy()
clone['TreeMetRate'] = clone['TreeMetRate'].astype(float)

site = pd.read_csv(IN_SITE) if IN_SITE.exists() else None
metadata_03 = json.loads(IN_METADATA.read_text()) if IN_METADATA.exists() else {}

n_loy = int((clone['enriched'] == 'LOY').sum())
n_roy = int((clone['enriched'] == 'ROY').sum())
print(f'Loaded {clone.shape[0]} LOY/ROY-enriched clones (LOY={n_loy}, ROY={n_roy}).')
if site is not None:
    print(f'Loaded site-level table: {site.shape[0]} rows')
print('Notebook 03 S6H metadata:', {k: metadata_03[k] for k in sorted(metadata_03) if 'S6H' in k or 'THRESH' in k or 'threshold' in k})

In [ ]:
# Current S6H production cutoffs and historical lineage references
CURRENT_WEAK = 0.001
CURRENT_HIGH = 0.008
HISTORICAL_LINEAGE_WEAK = 0.02
HISTORICAL_LINEAGE_HIGH = 0.10
EXPECTED_HIGH_RANGE = (20.0, 25.0)

WEAK_GRID = [0.001, 0.003, 0.005, 0.008, 0.01, 0.015, 0.02, 0.03, 0.04]
HIGH_GRID = [0.008, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.12, 0.15]

CANDIDATE_CUTOFFS = [
    {'label': 'Current', 'weak_cutoff': CURRENT_WEAK, 'high_cutoff': CURRENT_HIGH, 'kind': 'production'},
    {'label': 'Historical default', 'weak_cutoff': HISTORICAL_LINEAGE_WEAK, 'high_cutoff': HISTORICAL_LINEAGE_HIGH, 'kind': 'historical_default'},
    {'label': 'Stricter alternative', 'weak_cutoff': 0.03, 'high_cutoff': 0.10, 'kind': 'historical_stricter'},
]

REQUIRE_ALL_THREE_CATEGORIES_NONZERO = False
MAX_ROWS_TO_DISPLAY = 20

print('Current production cutoffs:', {'weak': CURRENT_WEAK, 'high': CURRENT_HIGH})
print('Historical lineage references:', {'weak': HISTORICAL_LINEAGE_WEAK, 'high': HISTORICAL_LINEAGE_HIGH})
print('Expected overall Highly metastatic range:', EXPECTED_HIGH_RANGE)

In [ ]:
def assign_pheno(tree_met_rate: float, weak_cutoff: float, high_cutoff: float) -> str:
    if tree_met_rate >= high_cutoff:
        return 'Highly metastatic'
    if tree_met_rate >= weak_cutoff:
        return 'Weakly metastatic'
    return 'Non metastatic'


def distance_to_range(value: float, low: float, high: float) -> float:
    if low <= value <= high:
        return 0.0
    return min(abs(value - low), abs(value - high))


def evaluate_cutoffs(df: pd.DataFrame, weak_cutoff: float, high_cutoff: float) -> tuple[dict, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    x = df.copy()
    x['metastatic_phenotype'] = x['TreeMetRate'].map(lambda v: assign_pheno(float(v), weak_cutoff, high_cutoff))

    tab = x.groupby(['enriched', 'metastatic_phenotype'], observed=False).size().unstack(fill_value=0)
    for c in ['Non metastatic', 'Weakly metastatic', 'Highly metastatic']:
        if c not in tab.columns:
            tab[c] = 0
    tab = tab[['Non metastatic', 'Weakly metastatic', 'Highly metastatic']]
    tab = tab.reindex(['LOY', 'ROY']).fillna(0)

    pct = tab.div(tab.sum(axis=1).replace(0, np.nan), axis=0) * 100
    pct_long = pct.reset_index().melt(id_vars='enriched', var_name='metastatic_phenotype', value_name='percent')

    hi_loy = int(tab.loc['LOY', 'Highly metastatic'])
    hi_roy = int(tab.loc['ROY', 'Highly metastatic'])
    other_loy = int(tab.loc['LOY'].sum() - hi_loy)
    other_roy = int(tab.loc['ROY'].sum() - hi_roy)
    odds_high, p_high = fisher_exact([[hi_loy, other_loy], [hi_roy, other_roy]])

    met_loy = int(tab.loc['LOY', 'Weakly metastatic'] + tab.loc['LOY', 'Highly metastatic'])
    met_roy = int(tab.loc['ROY', 'Weakly metastatic'] + tab.loc['ROY', 'Highly metastatic'])
    non_loy = int(tab.loc['LOY', 'Non metastatic'])
    non_roy = int(tab.loc['ROY', 'Non metastatic'])
    odds_any, p_any = fisher_exact([[met_loy, non_loy], [met_roy, non_roy]])

    try:
        _, p_chi2, _, _ = chi2_contingency(tab.values)
    except Exception:
        p_chi2 = np.nan

    overall_counts = tab.sum(axis=0)
    overall_total = int(overall_counts.sum())
    overall_high_pct = float(overall_counts['Highly metastatic'] / overall_total * 100) if overall_total else np.nan
    overall_weak_pct = float(overall_counts['Weakly metastatic'] / overall_total * 100) if overall_total else np.nan
    overall_non_pct = float(overall_counts['Non metastatic'] / overall_total * 100) if overall_total else np.nan

    out = {
        'weak_cutoff': weak_cutoff,
        'high_cutoff': high_cutoff,
        'LOY_non_pct': float(pct.loc['LOY', 'Non metastatic']),
        'LOY_weak_pct': float(pct.loc['LOY', 'Weakly metastatic']),
        'LOY_high_pct': float(pct.loc['LOY', 'Highly metastatic']),
        'ROY_non_pct': float(pct.loc['ROY', 'Non metastatic']),
        'ROY_weak_pct': float(pct.loc['ROY', 'Weakly metastatic']),
        'ROY_high_pct': float(pct.loc['ROY', 'Highly metastatic']),
        'overall_non_pct': overall_non_pct,
        'overall_weak_pct': overall_weak_pct,
        'overall_high_pct': overall_high_pct,
        'within_expected_high_range': int(EXPECTED_HIGH_RANGE[0] <= overall_high_pct <= EXPECTED_HIGH_RANGE[1]),
        'distance_to_expected_high_range': float(distance_to_range(overall_high_pct, EXPECTED_HIGH_RANGE[0], EXPECTED_HIGH_RANGE[1])),
        'delta_high_pct_LOY_minus_ROY': float(pct.loc['LOY', 'Highly metastatic'] - pct.loc['ROY', 'Highly metastatic']),
        'fisher_high_vs_others_p': float(p_high),
        'fisher_anymet_vs_non_p': float(p_any),
        'fisher_high_vs_others_odds': float(odds_high),
        'fisher_anymet_vs_non_odds': float(odds_any),
        'chi2_2x3_p': float(p_chi2),
        'all_three_nonzero_both_groups': int((tab > 0).all(axis=1).all()),
        'LOY_n': int(tab.loc['LOY'].sum()),
        'ROY_n': int(tab.loc['ROY'].sum()),
        'all_n': overall_total,
    }
    return out, tab, pct, pct_long

In [ ]:
rows = []
for weak in WEAK_GRID:
    for high in HIGH_GRID:
        if high <= weak:
            continue
        row, _, _, _ = evaluate_cutoffs(clone, weak, high)
        rows.append(row)

sweep = pd.DataFrame(rows)
sweep['minus_log10_chi2_p'] = -np.log10(sweep['chi2_2x3_p'].clip(lower=1e-300))
sweep['minus_log10_fisher_high_p'] = -np.log10(sweep['fisher_high_vs_others_p'].clip(lower=1e-300))

out_csv = DIR_PLOT_DATA / 'FigS6H_threshold_sweep.csv'
sweep.to_csv(out_csv, index=False)
print('[Saved]', out_csv)
print('rows:', sweep.shape[0])

In [ ]:
show = sweep.copy()
if REQUIRE_ALL_THREE_CATEGORIES_NONZERO:
    show = show[show['all_three_nonzero_both_groups'] == 1]

cols = [
    'weak_cutoff', 'high_cutoff',
    'LOY_non_pct', 'LOY_weak_pct', 'LOY_high_pct',
    'ROY_non_pct', 'ROY_weak_pct', 'ROY_high_pct',
    'overall_non_pct', 'overall_weak_pct', 'overall_high_pct',
    'within_expected_high_range', 'distance_to_expected_high_range',
    'delta_high_pct_LOY_minus_ROY',
    'fisher_high_vs_others_p', 'fisher_anymet_vs_non_p', 'chi2_2x3_p'
]

print('Top by chi2_2x3_p:')
display(show.sort_values('chi2_2x3_p').head(MAX_ROWS_TO_DISPLAY)[cols])

print('Top by fisher_high_vs_others_p:')
display(show.sort_values('fisher_high_vs_others_p').head(MAX_ROWS_TO_DISPLAY)[cols])

print('Top by closeness to expected overall highly-metastatic range:')
display(show.sort_values(['distance_to_expected_high_range', 'fisher_high_vs_others_p']).head(MAX_ROWS_TO_DISPLAY)[cols])

print('Current production row:')
display(show[(show['weak_cutoff'] == CURRENT_WEAK) & (show['high_cutoff'] == CURRENT_HIGH)][cols])

print('Historical lineage reference row (if present in current grid):')
display(show[(show['weak_cutoff'] == HISTORICAL_LINEAGE_WEAK) & (show['high_cutoff'] == HISTORICAL_LINEAGE_HIGH)][cols])

## Candidate comparison

The rows below are the discussion-focused candidates to compare visually in the S6H stacked-bar panel:

- current production setting: `0.001 / 0.008`
- historical default: `0.02 / 0.10`
- stricter alternative: `0.03 / 0.10`

The `overall_high_pct` column is included only as a soft reference for the collaborator expectation that `Highly metastatic` may fall around 20-25%.

In the current sweep, the settings that land in this expected range consistently use `high_cutoff = 0.10`. This suggests that the main driver of the `Highly metastatic` fraction is the `high` cutoff, while the `weak` cutoff mainly redistributes clones between `Non` and `Weakly metastatic`.

This discussion-focused comparison intentionally hides the legacy-like `0.001 / 0.02` and `0.001 / 0.03` settings from the panel preview, because they are still useful for sweep context but not ideal for collaborator-facing decision making.


In [ ]:
candidate_rows = []
candidate_pct_long = []
candidate_counts = {}
candidate_pct = {}

for cfg in CANDIDATE_CUTOFFS:
    stats, counts, pct, pct_long = evaluate_cutoffs(clone, cfg['weak_cutoff'], cfg['high_cutoff'])
    row = {**cfg, **stats}
    candidate_rows.append(row)

    counts_out = counts.copy()
    counts_out.insert(0, 'candidate_label', cfg['label'])
    counts_out.insert(1, 'candidate_kind', cfg['kind'])
    counts_out.insert(2, 'weak_cutoff', cfg['weak_cutoff'])
    counts_out.insert(3, 'high_cutoff', cfg['high_cutoff'])
    candidate_counts[cfg['label']] = counts_out

    pct_out = pct.copy()
    pct_out.insert(0, 'candidate_label', cfg['label'])
    pct_out.insert(1, 'candidate_kind', cfg['kind'])
    pct_out.insert(2, 'weak_cutoff', cfg['weak_cutoff'])
    pct_out.insert(3, 'high_cutoff', cfg['high_cutoff'])
    candidate_pct[cfg['label']] = pct_out

    pct_long = pct_long.copy()
    pct_long.insert(0, 'candidate_label', cfg['label'])
    pct_long.insert(1, 'candidate_kind', cfg['kind'])
    pct_long.insert(2, 'weak_cutoff', cfg['weak_cutoff'])
    pct_long.insert(3, 'high_cutoff', cfg['high_cutoff'])
    candidate_pct_long.append(pct_long)

candidate_summary = pd.DataFrame(candidate_rows)
candidate_pct_long_df = pd.concat(candidate_pct_long, ignore_index=True)

candidate_summary = candidate_summary[[
    'label', 'kind', 'weak_cutoff', 'high_cutoff',
    'LOY_non_pct', 'LOY_weak_pct', 'LOY_high_pct',
    'ROY_non_pct', 'ROY_weak_pct', 'ROY_high_pct',
    'overall_non_pct', 'overall_weak_pct', 'overall_high_pct',
    'within_expected_high_range', 'distance_to_expected_high_range',
    'delta_high_pct_LOY_minus_ROY',
    'fisher_high_vs_others_p', 'fisher_anymet_vs_non_p', 'chi2_2x3_p'
]].sort_values(['distance_to_expected_high_range', 'fisher_high_vs_others_p', 'chi2_2x3_p']).reset_index(drop=True)

candidate_summary.to_csv(DIR_PLOT_DATA / 'FigS6H_candidate_cutoff_summary.csv', index=False)
candidate_pct_long_df.to_csv(DIR_PLOT_DATA / 'FigS6H_candidate_cutoff_pct_long.csv', index=False)
for label, counts_df in candidate_counts.items():
    safe = label.lower().replace(' ', '_')
    counts_df.to_csv(DIR_PLOT_DATA / f'FigS6H_candidate_counts_{safe}.csv')
for label, pct_df in candidate_pct.items():
    safe = label.lower().replace(' ', '_')
    pct_df.to_csv(DIR_PLOT_DATA / f'FigS6H_candidate_pct_{safe}.csv')

display(candidate_summary)

In [ ]:
pheno_stack_order = ['Highly metastatic', 'Weakly metastatic', 'Non metastatic']
pheno_legend_order = ['Non metastatic', 'Weakly metastatic', 'Highly metastatic']
pheno_colors = {'Non metastatic': '#e5e5e5', 'Weakly metastatic': 'salmon', 'Highly metastatic': 'darkred'}

for cfg in CANDIDATE_CUTOFFS:
    _, _, pct, _ = evaluate_cutoffs(clone, cfg['weak_cutoff'], cfg['high_cutoff'])
    fig, ax = plt.subplots(figsize=(3.2, 3.6), dpi=260)
    bottom = np.zeros(len(pct.index))
    x = np.arange(len(pct.index))
    for phenotype in pheno_stack_order:
        vals = pct[phenotype].fillna(0).values
        ax.bar(x, vals, bottom=bottom, color=pheno_colors[phenotype], label=phenotype)
        bottom += vals

    labels = [f'LOY\n(n={n_loy})', f'ROY\n(n={n_roy})']
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Percentage of clones')
    ax.set_ylim(0, 100)
    ax.set_title(f"{cfg['label']}\nweak={cfg['weak_cutoff']:g}, high={cfg['high_cutoff']:g}")

    handles, labels0 = ax.get_legend_handles_labels()
    handle_map = {label: handle for handle, label in zip(handles, labels0)}
    ax.legend([handle_map[label] for label in pheno_legend_order if label in handle_map],
              [label for label in pheno_legend_order if label in handle_map],
              title='Metastatic phenotype',
              frameon=False,
              bbox_to_anchor=(1.02, 1),
              loc='upper left')
    fig.tight_layout()
    safe = cfg['label'].lower().replace(' ', '_')
    out_fig = DIR_FIG_SAVE / f'FigS6H_preview_{safe}_weak_{cfg["weak_cutoff"]:g}_high_{cfg["high_cutoff"]:g}.pdf'
    my_savefig(out_fig)
    plt.show()
    plt.close(fig)

In [ ]:
fig, axes = plt.subplots(1, len(CANDIDATE_CUTOFFS), figsize=(3.8 * len(CANDIDATE_CUTOFFS), 3.8), dpi=260, sharey=True)
if len(CANDIDATE_CUTOFFS) == 1:
    axes = [axes]

for ax, cfg in zip(axes, CANDIDATE_CUTOFFS):
    stats, _, pct, _ = evaluate_cutoffs(clone, cfg['weak_cutoff'], cfg['high_cutoff'])
    bottom = np.zeros(len(pct.index))
    x = np.arange(len(pct.index))
    for phenotype in pheno_stack_order:
        vals = pct[phenotype].fillna(0).values
        ax.bar(x, vals, bottom=bottom, color=pheno_colors[phenotype], label=phenotype)
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels([f'LOY\n(n={n_loy})', f'ROY\n(n={n_roy})'])
    ax.set_ylim(0, 100)
    ax.set_title(
        f"{cfg['label']}\nweak={cfg['weak_cutoff']:g}, high={cfg['high_cutoff']:g}\n"
        f"overall high={stats['overall_high_pct']:.1f}%"
    )
    if ax is axes[0]:
        ax.set_ylabel('Percentage of clones')

handles, labels0 = axes[-1].get_legend_handles_labels()
handle_map = {label: handle for handle, label in zip(handles, labels0)}
fig.legend([handle_map[label] for label in pheno_legend_order if label in handle_map],
           [label for label in pheno_legend_order if label in handle_map],
           title='Metastatic phenotype',
           frameon=False,
           bbox_to_anchor=(1.02, 1),
           loc='upper left')
fig.suptitle('Figure S6H cutoff candidates', y=1.05)
fig.tight_layout()
out_fig = DIR_FIG_SAVE / 'FigS6H_candidate_comparison.pdf'
my_savefig(out_fig)
plt.show()
plt.close(fig)

In [ ]:
metadata_out = {
    'input_clone_summary': str(IN_CLONE),
    'input_site_table': str(IN_SITE),
    'input_metadata_json': str(IN_METADATA),
    'n_loy_clones': n_loy,
    'n_roy_clones': n_roy,
    'current_production_cutoff': {'weak': CURRENT_WEAK, 'high': CURRENT_HIGH},
    'historical_lineage_reference': {'weak': HISTORICAL_LINEAGE_WEAK, 'high': HISTORICAL_LINEAGE_HIGH},
    'expected_overall_high_range': {'min': EXPECTED_HIGH_RANGE[0], 'max': EXPECTED_HIGH_RANGE[1]},
    'candidate_cutoffs': CANDIDATE_CUTOFFS,
    'weak_grid': WEAK_GRID,
    'high_grid': HIGH_GRID,
}
metadata_path = DIR_PLOT_DATA / 'FigS6H_threshold_exploration_metadata.json'
metadata_path.write_text(json.dumps(metadata_out, indent=2))
print('[Saved]', metadata_path)